In [2]:
%cd /content/drive/MyDrive/Colab\ Notebooks/transformer

/content/drive/MyDrive/Colab Notebooks/transformer


In [3]:
import os
print(os.getcwdb())

b'/content/drive/MyDrive/Colab Notebooks/transformer'


# **1. Central configuration for the Transformer**

In [4]:
%%writefile config.py
"""
config.py - Central configuration for the Transformer.
Every single hyperparameter lives here.
Architecture matches "Attention Is All You Need" (Vaswani et al., 2017) base model.
"""

from dataclasses import dataclass
import torch

@dataclass
class TransformerConfig:
    """
    Holds all hyperparameters for the Transformer model and training.

    ARCHITECTURE HYPERPARAMETERS
    ----------------------------
    d_model:     Dimension of all embeddings and sublayer inputs/outputs throughout the model.
    n_heads:     Number of attention heads in multi-head attention. Must evenly divide d_model (d_k = d_model / n_heads).
    n_layers:    Number of encoder layers AND decoder layers.
    d_ff:        Dimension of the inner (hidden) layer in the position-wise feed-forward networks. Paper: 2048 (= 4 * d_model).
    dropout:     Dropout probability. Applied after each sub-layer output, and to embedding + positional encoding sums. Paper: 0.1 for base model.
    max_seq_len: Maximum sequence length the model can handle. Positional encoding is precomputed up to this length.
    vocab_size:  Size of the shared source+target vocabulary.

    DERIVED DIMENSIONS (computed automatically)
    -------------------------------------------
    d_k:         Dimension of queries and keys per head = d_model / n_heads. Paper: 64.
    d_v:         Dimension of values per head = d_model / n_heads. Paper: 64.

    TRAINING HYPERPARAMETERS
    ------------------------
    batch_size:      Number of training examples per gradient update.
    num_epochs:      Total number of passes over the training set.
    warmup_steps:    Number of steps over which to linearly increase the learning rate before decaying.
    label_smoothing: Epsilon for label smoothing. A small amount of probability mass is spread to all tokens. Paper: 0.1
    PAD_IDX:         Token ID used for padding sequences to equal length. We use 0. This must match the dataset's tokenizer.
    SOS_IDX:         Start-of-sequence token ID. Prepended to decoder input.
    EOS_IDX:         End-of-sequence token ID. Marks end of output.

    DEVICE
    ------
    device:      'cuda' if GPU available, else 'cpu'. Auto-detected.
    """

    # Architecture
    d_model: int = 512
    n_heads: int = 8
    n_layers: int = 6
    d_ff: int = 2048
    # dropout: float = 0.1
    dropout: float = 0.2
    # max_seq_len: int = 100
    # vocab_size: int = 1000
    max_seq_len: int = 128      # <--- Locked in after data analysis
    vocab_size: int = 37000     # <--- Locked in to match the trained BPE

    # Derived
    @property
    def d_k(self) -> int:
        """Dimension of queries and keys per attention head."""
        assert self.d_model % self.n_heads == 0, (
            f"d_model ({self.d_model}) must be divisible by n_heads ({self.n_heads})"
        )
        return self.d_model // self.n_heads

    @property
    def d_v(self) -> int:
        """Dimension of values per attention head (same as d_k in this paper)."""
        return self.d_model // self.n_heads

    # Training
    batch_size: int = 32
    num_epochs: int = 30
    # warmup_steps: int = 4000
    warmup_steps: int = 2000
    label_smoothing: float = 0.1

    # Special tokens
    PAD_IDX: int = 0    # Padding token — sequences shorter than max are padded
    SOS_IDX: int = 1    # Start-of-sequence token — prepended to decoder input
    EOS_IDX: int = 2    # End-of-sequence token — appended to targets

    # Device
    @property
    def device(self) -> torch.device:
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def validate_config(config: TransformerConfig):
    """Run basic consistency checks on the configuration."""
    assert config.d_model > 0, "d_model must be positive"
    assert config.n_heads > 0, "n_heads must be positive"
    assert config.d_model % config.n_heads == 0, (
        f"d_model must be divisible by n_heads: {config.d_model} % {config.n_heads} != 0"
    )
    assert config.n_layers > 0, "n_layers must be positive"
    assert config.d_ff > 0, "d_ff must be positive"
    assert 0.0 <= config.dropout < 1.0, "dropout must be in [0, 1)"
    assert config.vocab_size > 3, "vocab_size must be > 3 (PAD, SOS, EOS need IDs)"
    assert config.max_seq_len > 0, "max_seq_len must be positive"
    print(f"[Config] Validated. d_model={config.d_model}, n_heads={config.n_heads}, "
          f"d_k={config.d_k}, n_layers={config.n_layers}, d_ff={config.d_ff}")
    print(f"[Config] Device: {config.device}")

# Singleton instance: We will import this throughout the project: `from config import cfg`
cfg = TransformerConfig()

Overwriting config.py


# **2. Token embeddings and sinusoidal positional encoding**

In [ ]:
%%writefile embeddings.py
"""
embeddings.py - Token embeddings and sinusoidal positional encoding.

TWO COMPONENTS:
  1. TokenEmbedding:        Maps integer token IDs -> dense vectors of dim d_model. Learned. Scaled by sqrt(d_model).
  2. PositionalEncoding:    Adds fixed sinusoidal vectors to encode position. Not learned (in the paper's version).
  3. TransformerEmbedding:  Combines both. The full embedding pipeline.

MATHEMATICAL DETAIL: Token Embedding
------------------------------------
Let E: (V, d_model) be the embedding matrix (V = vocab_size). For token ID k, embedding = E[k, :] - the k-th row of E.
For batch input x of shape [B, S] (B sequences, each S tokens):
  emb = E[x] -> shape [B, S, d_model], emb = emb * sqrt(d_model)  -> scaled

WHY SCALE: The embedding matrix is initialized with values ~ N(0, 1/d_model) (PyTorch default for nn.Embedding after we scale them). The positional
encodings have values in [-1, 1]. Without scaling embeddings up, the PE signal would dominate and the learned token meaning would be negligible.

MATHEMATICAL DETAIL: Positional Encoding
----------------------------------------
PE(pos, 2i)   = sin(pos / 10000^{2i / d_model})
PE(pos, 2i+1) = cos(pos / 10000^{2i / d_model})
For position `pos` and dimension index `i`:     frequency ω_i = 1 / (10000 ^ (2i / d_model))
The original paper applies dropout to the sum of embeddings + positional encodings. We will use that.
"""

import math
import torch
import torch.nn as nn
from config import cfg


class TokenEmbedding(nn.Module):
    """
    Learnable token embedding with sqrt(d_model) scaling.
    Wraps nn.Embedding, which is essentially a lookup table: nn.Embedding(vocab_size, d_model) stores a matrix of shape [vocab_size, d_model].

    Parameters (what get learned)
    -----------------------------
    weight: shape [vocab_size, d_model] - one vector per token in vocabulary.
    PyTorch's nn.Embedding initializes weights ~ N(0, 1) by default. We scale the output (not the weights themselves) by sqrt(d_model).
    """

    def __init__(self, vocab_size: int = cfg.vocab_size, d_model: int = cfg.d_model):
        super().__init__()
        self.d_model = d_model
        # nn.Embedding: a simple lookup table that stores embeddings of a fixed dictionary.
        # the embedding for PAD token is kept as zeros and receives no gradient updates
        self.embedding = nn.Embedding(
            vocab_size, d_model, padding_idx=cfg.PAD_IDX
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : shape [B, S], Integer token IDs. Each value must be in [0, vocab_size - 1].

        Returns
        -------
        emb : [B, S, d_model], Scaled embedding vectors.
        """
        return self.embedding(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    """
    Fixed sinusoidal positional encoding.

    Not learned - the encoding is computed once at initialization using the formulas from the paper and stored as a non-parameter buffer.
    During forward, it's simply added to the input embeddings.

    BUFFER vs PARAMETER:
    - nn.Parameter: included in model.parameters(), gets gradients, updates
    - register_buffer: part of model state (saved/loaded), but no gradients
    We use register_buffer because PE is fixed - it never changes.
    """

    def __init__(self, d_model: int = cfg.d_model, max_seq_len: int = cfg.max_seq_len, dropout: float = cfg.dropout):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Construct the PE matrix of shape [max_seq_len, d_model]
        # pe[pos, i] = sin(pos / 10000^(2*(i//2) / d_model)) if i is even
        #            = cos(pos / 10000^(2*(i//2) / d_model)) if i is odd

        # Initialize with zeros; we'll fill it in
        pe = torch.zeros(max_seq_len, d_model)  # shape: [max_seq_len, d_model]

        # position: column vector of positions [0, 1, 2, ..., max_seq_len-1]
        # shape: [max_seq_len, 1]
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)

        # div_term: the denominators 10000^(2i/d_model) for i = 0, 1, ..., d_model//2-1
        # We compute this in log space for numerical stability:
        #   10000^(2i/d_model) = exp(2i/d_model * log(10000))
        # This gives shape [d_model // 2]
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() *   # [0, 2, 4, ..., d_model-2]
            (-math.log(10000.0) / d_model)           # scale factor
        )
        # div_term[i] = exp(-2i/d_model * log(10000)) = 1 / 10000^(2i/d_model)

        # Fill even dimensions with sin, odd with cos
        pe[:, 0::2] = torch.sin(position * div_term)  # dims 0,2,4,...
        pe[:, 1::2] = torch.cos(position * div_term)  # dims 1,3,5,...

        # Add batch dimension: [max_seq_len, d_model] -> [1, max_seq_len, d_model] # So it broadcasts with inputs of shape [B, S, d_model]
        pe = pe.unsqueeze(0)

        # Register as a buffer (not a parameter - won't be updated by optimizer)
        self.register_buffer('pe', pe)
        # After this: self.pe has shape [1, max_seq_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Add positional encoding to input and apply dropout.

        Parameters
        ---------
        x : [B, S, d_model], Token embeddings.

        Returns
        ------
        x : [B, S, d_model], Embeddings with positional information added.
        """
        # self.pe[:, :x.size(1), :] slices PE to length S
        # The addition broadcasts: [B, S, 512] + [1, S, 512] -> [B, S, 512]
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class TransformerEmbedding(nn.Module):
    """
    Complete embedding pipeline: token embedding + positional encoding.
    This is the first thing the encoder and decoder apply to their inputs.
    Pipeline:
      token_ids [B, S] -> TokenEmbedding -> [B, S, d_model] -> PositionalEncoding → [B, S, d_model]

    WEIGHT SHARING:
    The paper shares the embedding weight matrix between:
      1. Source token embedding (encoder input)
      2. Target token embedding (decoder input)
      3. Pre-softmax linear projection (decoder output)
    This reduces parameters. In our Transformer class, we pass the same TokenEmbedding to both encoder and decoder embeddings.
    """

    def __init__(self, vocab_size: int = cfg.vocab_size, d_model: int = cfg.d_model,
                 max_seq_len: int = cfg.max_seq_len,
                 dropout: float = cfg.dropout):
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_len, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ---------
        x : [B, S], Token IDs.

        Returns
        -------
        out : [B, S, d_model], Embedding + positional encoding with dropout.
        """
        # 1. Lookup embeddings and scale
        tok_emb = self.token_embedding(x)     # [B, S, d_model]
        # 2. Add positional encodings + dropout
        out = self.positional_encoding(tok_emb)  # [B, S, d_model]
        return out

Overwriting embeddings.py


# **3. Padding mask and look-ahead (causal) mask**

In [ ]:
%%writefile masks.py
"""
masks.py - Padding mask and look-ahead (causal) mask construction.

WHY MASKS EXIST
--------------
The Transformer's attention mechanism, in its raw form, allows every position
to attend to every other position. We need to selectively "block" certain
attention connections for two reasons:

  1. PADDING MASK: Sequences in a batch have different lengths. Shorter ones
     are right-padded with PAD tokens (ID = 0). We don't want any position to
     attend to a padding token, it carries no meaning and would corrupt the
     representation.

  2. LOOK-AHEAD (CAUSAL) MASK: In the decoder's self-attention, when predicting
     the token at position t, the model must not be able to "see" tokens at
     positions t+1, t+2, ... These are future tokens - not yet generated. Allowing
     the model to see them during training would be "cheating" and would prevent
     it from learning a proper generative model. This mask enforces autoregressive
     behavior.

HOW MASKS WORK
-------------
After computing the raw attention scores (shape [B, H, S_q, S_k]):
  scores = scores.masked_fill(mask == True, float('-inf'))

Where mask has value True at positions to BLOCK. Then:

  attention_weights = softmax(scores, dim=-1)

Since exp(-inf) = 0, masked positions receive exactly 0 attention weight.
They are completely invisible to the attending position.

SHAPE CONVENTIONS
-----------------
Attention scores are shape [B, H, S_q, S_k]:
  B = batch size
  H = number of heads
  S_q = query sequence length
  S_k = key sequence length

Masks must be broadcastable to this shape.

  Padding mask: [B, 1, 1, S_k]
    - We don't differentiate between heads (broadcast over H)
    - We don't differentiate between query positions (broadcast over S_q)
    - We only care about which KEY positions to mask

  Look-ahead mask: [1, 1, S_q, S_k]  (or [S_q, S_k])
    - Same for all batch elements and heads
    - Upper-triangular: masks future key positions for each query position

  Combined decoder mask: max(look_ahead_mask, padding_mask)
    - True wherever either mask says to block
"""

import torch


def make_padding_mask(seq: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """
    Create a padding mask that blocks attention to PAD tokens.

    Parameters
    ----------
    seq : shape [B, S], Token ID sequences. PAD positions have value pad_idx.
    pad_idx : int, The token ID used for padding. Default: 0.

    Returns
    -------
    mask : shape [B, 1, 1, S]
        Boolean tensor. True = "mask this position" (it's a PAD token).
        Shape is expanded for broadcasting over (heads, query positions).

    Example
    -------
    seq = [[45, 732, 1, 0, 0],   # 3 real tokens + 2 PAD
           [12, 74, 395, 2, 100]] # 5 real tokens
    mask = [[[[False, False, False, True, True]]],
            [[[False, False, False, False, False]]]]
    Shape: [2, 1, 1, 5]

    When broadcast against scores [2, 8, 5, 5], this blocks attention TO positions 3 and 4 in the first sequence, for all 8 heads and
    all 5 query positions so that query do not attend to padding tokens.
    """
    # seq shape: [B, S]
    # unsqueeze(1) -> [B, 1, S] -> unsqueeze(2) -> [B, 1, 1, S] <- broadcasts over heads and query positions
    mask = (seq == pad_idx).unsqueeze(1).unsqueeze(2)
    return mask


def make_causal_mask(seq_len: int, device: torch.device = None) -> torch.Tensor:
    """
    Create a causal (look-ahead) mask for the decoder self-attention.
    Blocks position i from attending to position j when j > i (future).

    Parameters
    ----------
    seq_len : int, Length of the sequence (S).
    device : torch.device, Device to create the mask on.

    Returns
    -------
    mask : shape [1, 1, S, S], Boolean tensor. True = "mask this (i,j) pair."
    Visualization (S=5):
    --------------------

    Q-K  0     1     2     3     4
    0  [F     T     T     T     T] <- pos 0 can only attend to pos 0
    1  [F     F     T     T     T] <- pos 1 can attend to pos 0,1
    2  [F     F     F     T     T] <- pos 2 can attend to pos 0,1,2
    3  [F     F     F     F     T]
    4  [F     F     F     F     F] <- pos 4 can attend to all

    T = True = masked (blocked)
    F = False = allowed

    torch.triu with diagonal=1 gives the strict upper triangle.
    """
    mask = torch.triu(
        torch.ones(seq_len, seq_len, dtype=torch.bool, device=device),
        diagonal=1
    )
    # shape: [S, S] -> [1, 1, S, S] for broadcasting over (batch, heads)
    return mask.unsqueeze(0).unsqueeze(0)


def make_src_mask(src: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """
    Create source padding mask for the encoder.

    Used in:
    - Encoder self-attention: blocks attention TO padding positions.
    - Decoder cross-attention: same - blocks attention to padding in the source.

    Parameters
    ----------
    src : shape [B, S_src]
    pad_idx : int

    Returns
    -------
    mask : shape [B, 1, 1, S_src]
    """
    return make_padding_mask(src, pad_idx)


def make_tgt_mask(tgt: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """
    Create combined target mask for the decoder self-attention.

    Combines:
    1. Causal mask: prevents attending to future positions.
    2. Padding mask: prevents attending to PAD tokens.

    The combined mask is the logical OR - True if EITHER condition applies.

    Parameters
    ----------
    tgt : shape [B, S_tgt]
    pad_idx : int

    Returns
    -------
    mask : shape [B, 1, S_tgt, S_tgt]
        True = masked (blocked).
    """
    S_tgt = tgt.size(1)
    device = tgt.device

    # Causal mask: shape [1, 1, S_tgt, S_tgt]
    causal = make_causal_mask(S_tgt, device=device)

    # Padding mask: shape [B, 1, 1, S_tgt]
    padding = make_padding_mask(tgt, pad_idx)

    # Combine: torch.logical_or broadcasts [B, 1, 1, S] with [1, 1, S, S]
    # Result: [B, 1, S_tgt, S_tgt]
    combined = torch.logical_or(causal, padding)

    return combined

Overwriting masks.py


# **4. Scaled Dot-Product Attention and Multi-Head Attention**

In [ ]:
%%writefile attention.py
"""
attention.py - Scaled Dot-Product Attention and Multi-Head Attention.

This file contains the core computational engine of the Transformer.

COMPONENTS:
  1. scaled_dot_product_attention() - The fundamental attention operation. A function (not a module) - it's pure computation with no parameters.
  2. MultiHeadAttention - The full multi-head attention module with learned projection matrices W_Q, W_K, W_V, W_O.

USAGE IN THE TRANSFORMER:
  - Encoder self-attention:       Q=K=V=encoder_input
  - Decoder masked self-attention: Q=K=V=decoder_input (with causal mask)
  - Decoder cross-attention:       Q=decoder_state, K=V=encoder_output

The SAME MultiHeadAttention class is going to handle all three cases -
the difference is just which tensors are passed as Q, K, V.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import cfg


def scaled_dot_product_attention(
    Q: torch.Tensor,
    K: torch.Tensor,
    V: torch.Tensor,
    mask: torch.Tensor = None,
    dropout: nn.Dropout = None
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute scaled dot-product attention.
    Formula:
        Attention(Q, K, V) = softmax(Q.K^T / sqrt(d_k)).V

    Parameters
    ---------
    Q : shape -> [B, H, S_q, d_k]
        Queries. B=batch, H=heads, S_q=query length, d_k=key/query dim.
    K : shape -> [B, H, S_k, d_k]
        Keys. S_k=key/value length (same as S_q for self-attention).
    V : shape -> [B, H, S_k, d_v]
        Values. d_v=value dim (equal to d_k in this paper).
    mask : torch.Tensor or None, shape broadcastable to [B, H, S_q, S_k]
        Boolean mask. True = "block this position" -> set score to -inf.
        - Padding mask:   shape [B, 1, 1, S_k]   — broadcast over H and S_q
        - Causal mask:    shape [1, 1, S_q, S_k] — broadcast over B and H
        - Combined:       shape [B, 1, S_q, S_k]
    dropout : nn.Dropout or None
        If provided, applied to the attention weights (after softmax). Paper: dropout applied to attention weights (Section 5.4).

    Returns
    -------
    output : torch.Tensor, shape [B, H, S_q, d_v]. Attention-weighted sum of values.
    attention_weights : torch.Tensor, shape [B, H, S_q, S_k]
        Normalized attention weights. Useful for visualization.
    """
    # d_k is the last dimension of Q (and K)
    d_k = Q.size(-1)

    # Step 1: Compute raw attention scores
    # Q shape:  [B, H, S_q, d_k]
    # K.T shape: [B, H, d_k, S_k]  (transpose last two dims)
    # scores shape: [B, H, S_q, S_k]
    # scores[b, h, i, j] = dot product of query i with key j (in head h, batch b)
    scores = torch.matmul(Q, K.transpose(-2, -1))
    # shape: [B, H, S_q, S_k]

    # Step 2: Scale by 1/sqrt(d_k)
    # Without this, dot products have std deviation sqrt(d_k), pushing softmax
    # into saturated regions with near-zero gradients.
    scores = scores / math.sqrt(d_k)
    # shape unchanged: [B, H, S_q, S_k]

    # Step 3: Apply mask
    # Where mask is True, set score to -inf so softmax gives 0 there.
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    # shape unchanged: [B, H, S_q, S_k]

    # Step 4: Softmax over the key dimension
    # For each query position, compute a probability distribution over all key positions.
    # dim=-1 means "normalize over the last dimension" = the S_k dimension.
    attention_weights = F.softmax(scores, dim=-1) # shape: [B, H, S_q, S_k] # Each row (for each query) sums to 1.
    # If a position was masked (-inf), its softmax output is 0.

    # NUMERICAL NOTE: If an entire row is -inf (e.g., a completely masked # query — which can happen for PAD queries), softmax produces NaN.
    # We handle this by replacing NaN with 0.
    attention_weights = torch.nan_to_num(attention_weights, nan=0.0)

    # Step 5: Apply dropout to attention weights # Paper: "We apply dropout to the output of the softmax function."
    if dropout is not None:
        attention_weights = dropout(attention_weights)

    # Step 6: Weighted sum of values
    # attention_weights: [B, H, S_q, S_k]
    # V:                 [B, H, S_k, d_v]
    # output:            [B, H, S_q, d_v]
    output = torch.matmul(attention_weights, V) # shape: [B, H, S_q, d_v]

    return output, attention_weights


class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention as described in Section 3.2.2.

    Instead of a single attention function with d_model-dimensional Q, K, V,
    we:
      1. Project Q, K, V to d_k, d_k, d_v dimensions for each of h heads (using learned linear projections - different per head)
      2. Run h scaled dot-product attention operations in parallel
      3. Concatenate the h outputs (shape: h * d_v = d_model)
      4. Apply a final output projection W_O

    By projecting to lower-dimensional subspaces, each head can
    attend to different aspects of the input simultaneously. Head 1 might
    focus on syntactic relationships, Head 2 on semantic similarity, etc.

    PARAMETERS (what gets learned):
    -------------------------------
    W_Q: shape [d_model, d_model] — projects input to all head queries at once
    W_K: shape [d_model, d_model] — projects input to all head keys at once
    W_V: shape [d_model, d_model] — projects input to all head values at once
    W_O: shape [d_model, d_model] — output projection (concatenated heads -> d_model)

    Note: W_Q, W_K, W_V each have shape [d_model, d_model] because they contain all h heads' projections stacked:
      [d_model, d_model] = [d_model, h * d_k]
    We split them into h heads in the forward pass.
    Total parameters: 4 * d_model^2 = 4 * 512^2 = 1,048,576 for base model.
    """

    def __init__(self, d_model: int = cfg.d_model, n_heads: int = cfg.n_heads, dropout: float = cfg.dropout):
        super().__init__()

        # Validate: d_model must be divisible by n_heads
        assert d_model % n_heads == 0, (
            f"d_model ({d_model}) must be divisible by n_heads ({n_heads})"
        )

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads   # dimension per head for Q and K
        self.d_v = d_model // n_heads   # dimension per head for V

        # ------------ Linear projection layers ------------
        # W_Q, W_K, W_V: each maps [*, d_model] -> [*, d_model]
        # Internally, this is all h heads' projections combined.
        # We'll split into heads in the forward pass.
        # nn.Linear(in, out) has weight [out, in] and bias [out].
        self.W_Q = nn.Linear(d_model, d_model, bias=True)  # queries projection
        self.W_K = nn.Linear(d_model, d_model, bias=True)  # keys projection
        self.W_V = nn.Linear(d_model, d_model, bias=True)  # values projection
        self.W_O = nn.Linear(d_model, d_model, bias=True)  # output projection

        # Dropout on attention weights
        self.attn_dropout = nn.Dropout(p=dropout)

        # Store last attention weights for visualization/debugging
        self.attention_weights = None

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Split the last dimension into (n_heads, d_k) and transpose.

        This is how we "separate" the combined projection matrix into
        individual heads without actually having h separate projection matrices.

        Shape transformation:
          [B, S, d_model] -> [B, S, h, d_k] -> [B, h, S, d_k]

        Example:
          d_model=512, n_heads=8, d_k=64
          [2, 5, 512] -> [2, 5, 8, 64] -> [2, 8, 5, 64]
        """
        B, S, _ = x.shape
        # Reshape: treat the d_model dimension as (n_heads, d_k)
        x = x.view(B, S, self.n_heads, self.d_k)
        # Transpose to put head dimension before sequence dimension:
        # [B, S, n_heads, d_k] -> [B, n_heads, S, d_k]
        return x.transpose(1, 2)

    def combine_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Reverse of split_heads: merge the head and d_k dimensions.

        Shape transformation:
          [B, h, S, d_k] -> [B, S, h, d_k] -> [B, S, h*d_k] = [B, S, d_model]

        Example:
          [2, 8, 5, 64] -> [2, 5, 8, 64] -> [2, 5, 512]
        """
        B, _, S, _ = x.shape
        # Transpose back: [B, h, S, d_k] → [B, S, h, d_k]
        x = x.transpose(1, 2)
        # .contiguous() is needed before .view() if the tensor isn't contiguous
        # in memory after the transpose (which it typically isn't).
        x = x.contiguous()
        # Reshape: merge n_heads and d_k -> d_model
        return x.view(B, S, self.n_heads * self.d_k)

    def forward(
        self,
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
        mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Forward pass for Multi-Head Attention.

        Parameters
        ----------
        Q : shape [B, S_q, d_model]
            Query input. In self-attention, same as K and V.
            In cross-attention (decoder), this is the decoder state.
        K : shape [B, S_k, d_model]
            Key input. In self-attention, same as Q.
            In cross-attention, this is the encoder output.
        V : shape [B, S_k, d_model]
            Value input. Same as K in all cases in this paper.
        mask : torch.Tensor or None
            Attention mask. True = block. See masks.py for shapes.

        Returns
        -------
        output : shape [B, S_q, d_model]
            Same shape as Q input. Each position now has a contextual
            representation formed from attending to K/V.

        Complete Shape Trace (B=2, S_q=5, S_k=5, d_model=512, h=8, d_k=64):
        -------------------------------------------------------------------
        Q input:       [2, 5, 512]
        K input:       [2, 5, 512]
        V input:       [2, 5, 512]

        W_Q(Q):        [2, 5, 512]   (linear projection: each position projected)
        W_K(K):        [2, 5, 512]
        W_V(V):        [2, 5, 512]

        split_heads(Q_proj): [2, 8, 5, 64]  (8 heads, each with 64-dim queries)
        split_heads(K_proj): [2, 8, 5, 64]
        split_heads(V_proj): [2, 8, 5, 64]

        attention_output: [2, 8, 5, 64]  (output of scaled dot-product attention)

        combine_heads(): [2, 5, 512]  (concatenate all 8 heads)

        W_O:           [2, 5, 512]  (output projection)
        """

        # Step 1: Linear projections for Q, K, V
        # Project from d_model to d_model (containing all heads combined)
        # W_Q maps [B, S, d_model] -> [B, S, d_model]
        Q_proj = self.W_Q(Q)  # [B, S_q, d_model]
        K_proj = self.W_K(K)  # [B, S_k, d_model]
        V_proj = self.W_V(V)  # [B, S_k, d_model]

        # Step 2: Split into multiple heads
        # [B, S, d_model] -> [B, n_heads, S, d_k]
        Q_heads = self.split_heads(Q_proj)  # [B, n_heads, S_q, d_k]
        K_heads = self.split_heads(K_proj)  # [B, n_heads, S_k, d_k]
        V_heads = self.split_heads(V_proj)  # [B, n_heads, S_k, d_v]

        # Step 3: Scaled dot-product attention for all heads in parallel
        # PyTorch's matmul broadcasts over leading dimensions (B, n_heads),
        # so this runs attention for all batch elements and all heads at once.
        attn_output, self.attention_weights = scaled_dot_product_attention(
            Q_heads, K_heads, V_heads,
            mask=mask,
            dropout=self.attn_dropout
        )
        # attn_output shape: [B, n_heads, S_q, d_v]

        # Step 4: Combine heads
        # [B, n_heads, S_q, d_v] -> [B, S_q, n_heads * d_v] = [B, S_q, d_model]
        combined = self.combine_heads(attn_output)  # [B, S_q, d_model]

        # Step 5: Output projection
        # W_O: [B, S_q, d_model] -> [B, S_q, d_model]
        output = self.W_O(combined)

        return output

Overwriting attention.py


# **5. Position-wise Feed-Forward Network (FFN)**

In [ ]:
%%writefile feedforward.py
"""
feedforward.py : Position-wise Feed-Forward Network (FFN).

WHAT IT IS
---------
A small two-layer MLP (multi-layer perceptron) applied independently
and identically to each position in the sequence:

    FFN(x) = max(0, x.W_1 + b_1).W_2 + b_2

This is applied AFTER the attention sub-layer in both encoder and decoder.

WHY IT EXISTS
------------
After multi-head attention, each position has gathered information from
all other positions. The attention output is a weighted blend of value
vectors - it aggregates context, but the aggregation is a LINEAR operation
over the values.

The FFN then processes each position's context-enriched representation
nonlinearly. It adds expressive power: the network can "think about" what
it has collected, applying a nonlinear transformation.

The expanded hidden dimension (d_ff = 2048 >> d_model = 512) gives room
    for multiple "features" to be computed simultaneously.

WHY "POSITION-WISE"?
-------------------
The FFN is applied to each position INDEPENDENTLY - no mixing across positions.
The SAME weight matrices (W_1, W_2) are applied at every position.
This is equivalent to two 1x1 convolutions along the sequence.

Position mixing ONLY happens in the attention layers. The FFN is a
per-position transformation.

THE EXPANSION FACTOR 4x:
------------------------
d_ff = 4 * d_model (2048 = 4 * 512 in the paper).

PARAMETERS
----------
W_1: shape [d_model, d_ff]  = [512, 2048]
b_1: shape [d_ff]           = [2048]
W_2: shape [d_ff, d_model]  = [2048, 512]
b_2: shape [d_model]        = [512]
"""

import torch
import torch.nn as nn
from config import cfg


class PositionwiseFeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.

    The same weight matrices are applied to each position separately.
    PyTorch's nn.Linear handles this naturally when the input has shape
    [B, S, d_model] - it applies the linear transformation to the last
    dimension, independently for each (b, s) position.
    """

    def __init__(self, d_model: int = cfg.d_model, d_ff: int = cfg.d_ff,
                 dropout: float = cfg.dropout):
        """
        Parameters
        ----------
        d_model : int
            Input and output dimension. Both linear layers start and end here.
        d_ff : int
            Hidden (inner) dimension. Larger than d_model (typically 4X).
        dropout : float
            Dropout applied after the ReLU activation.
        """
        super().__init__()

        # First linear layer: d_model -> d_ff
        # Applied to last dimension: [B, S, d_model] -> [B, S, d_ff]
        self.linear1 = nn.Linear(d_model, d_ff)

        # Second linear layer: d_ff -> d_model
        # Projects back to model dimension: [B, S, d_ff] -> [B, S, d_model]
        self.linear2 = nn.Linear(d_ff, d_model)

        # ReLU activation: max(0, x)
        # Applied element-wise after linear1.
        self.relu = nn.ReLU()

        # Dropout for regularization (applied after ReLU, before linear2)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Apply position-wise FFN to input.

        Parameters
        ----------
        x : shape [B, S, d_model]

        Returns
        -------
        output : shape [B, S, d_model]
            Same shape as input.

        Step-by-step shape trace (B=2, S=5, d_model=512, d_ff=2048):
        ------------------------------------------------------------
        x:            [2, 5, 512]
        linear1(x):   [2, 5, 2048]   (expand: each position goes 512->2048)
        relu:         [2, 5, 2048]   (negatives zeroed out)
        dropout:      [2, 5, 2048]   (random zeros, training only)
        linear2:      [2, 5, 512]    (compress: each position goes 2048->512)
        """
        # Step 1: Expand with first linear layer
        # x: [B, S, d_model] -> [B, S, d_ff]
        x = self.linear1(x)

        # Step 2: ReLU nonlinearity
        x = self.relu(x)

        # Step 3: Dropout
        x = self.dropout(x)

        # Step 4: Project back to d_model
        # [B, S, d_ff] -> [B, S, d_model]
        x = self.linear2(x)

        return x

Overwriting feedforward.py


# **6. Encoder Layer and Full Encoder Stack**

In [ ]:
%%writefile encoder.py
"""
encoder.py - Encoder Layer and Full Encoder Stack.

THE ENCODER'S JOB
-----------------
The encoder takes the source sequence (e.g., an English sentence) and
produces a sequence of contextual representations - one vector per
source position. These representations capture:
  - The meaning of each token
  - Its relationship to every other token in the sequence
  - Its position in the sequence

These representations are then used by the decoder via cross-attention.

ENCODER LAYER STRUCTURE
-----------------------
Each of the N=6 encoder layers applies, in order:

  1. Multi-Head Self-Attention (MHSA):
       Every position can attend to every other position.
       Q = K = V = x (all from the same input)
       With source padding mask applied.

  2. Residual + LayerNorm:
       x = LayerNorm(x + MHSA(x))

  3. Position-wise Feed-Forward Network (FFN):
       Applied independently to each position.

  4. Residual + LayerNorm:
       x = LayerNorm(x + FFN(x))

POST-NORM vs PRE-NORM:
The paper uses post-norm (shown above): normalize AFTER adding residual.
Pre-norm (used in many modern models): normalize BEFORE applying sub-layer:
  x = x + SubLayer(LayerNorm(x))

We implement post-norm to match the paper. Both are available here.

DROPOUT PLACEMENT:
Paper: dropout is applied to the output of each sub-layer, before
it is added to the sub-layer input. So:
  x = LayerNorm(x + Dropout(SubLayer(x)))

FULL ENCODER STACK
------------------
N identical encoder layers (N=6 in the paper), stacked sequentially.
Output of layer i becomes input to layer i+1.
The final output is passed to every decoder layer as memory (keys and values
for cross-attention).
"""

import torch
import torch.nn as nn
from attention import MultiHeadAttention
from feedforward import PositionwiseFeedForward
from config import cfg


class EncoderLayer(nn.Module):
    """
    A single encoder layer with MHSA + FFN, both with residual + LayerNorm.

    PARAMETERS (per layer):
    ----------------------------------------------
    MHSA parameters:
      W_Q, W_K, W_V, W_O: each [d_model, d_model] -> 4 x 512^2 = 1,048,576

    FFN parameters:
      W_1: [d_model, d_ff] = [512, 2048] = 1,048,576
      b_1: [d_ff] = 2048
      W_2: [d_ff, d_model] = [2048, 512] = 1,048,576
      b_2: [d_model] = 512

    LayerNorm parameters (x2, one per sub-layer):
      Y-gamma (gain): [d_model] = 512 (initialized to 1)
      B-beta (bias): [d_model] = 512 (initialized to 0)

    Total per layer: ~3.15M parameters
    Total for 6 layers: ~18.9M parameters (just encoder)
    """

    def __init__(self, d_model: int = cfg.d_model, n_heads: int = cfg.n_heads,
                 d_ff: int = cfg.d_ff, dropout: float = cfg.dropout):
        super().__init__()

        # Sub-layer 1: Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)

        # Sub-layer 2: Position-wise FFN
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

        # Layer Normalization (post-norm: applied AFTER residual addition)
        # Two separate LayerNorm instances — one per sub-layer.
        # LayerNorm(d_model) normalizes over the last dimension (d_model).
        # It has learned parameters Y-gamma and B-beta of shape [d_model].
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Dropout: Applied to sub-layer output before adding to residual.
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Forward pass through one encoder layer.

        Parameters
        ----------
        x : shape [B, S, d_model], Input sequence representations.
        mask : torch.Tensor or None, shape [B, 1, 1, S]
            Source padding mask. True = don't attend to that position.

        Returns
        -------
        x : torch.Tensor, shape [B, S, d_model]
            Updated sequence representations (same shape as input).

        Shape trace (B=2, S=5, d_model=512):
        ------------------------------------
        Input x:                [2, 5, 512]

        === Sub-layer 1: Self-Attention ===
        residual = x:           [2, 5, 512]
        attn = MHSA(x, x, x):  [2, 5, 512]  (self-attention: Q=K=V=x)
        dropout(attn):          [2, 5, 512]
        residual + attn:        [2, 5, 512]
        norm1(...):             [2, 5, 512]

        === Sub-layer 2: FFN ===
        residual = x:           [2, 5, 512]
        ffn_out = FFN(x):       [2, 5, 512]
        dropout(ffn_out):       [2, 5, 512]
        residual + ffn_out:     [2, 5, 512]
        norm2(...):             [2, 5, 512]

        Output:                 [2, 5, 512]  <- same as input!
        """

        # === Sub-layer 1: Multi-Head Self-Attention ===
        # Save residual (the "skip connection" input)
        residual = x

        # Self-attention: Q = K = V = x. The encoder can attend to ALL positions (no causal masking needed here), only padding positions are masked.
        attn_output = self.self_attention(x, x, x, mask) # shape: [B, S, d_model]

        # Dropout + residual connection + layer normalization (post-norm)
        x = self.norm1(residual + self.dropout(attn_output)) # shape: [B, S, d_model]

        # === Sub-layer 2: Position-wise FFN ====
        residual = x
        ffn_output = self.feed_forward(x) # shape: [B, S, d_model]
        x = self.norm2(residual + self.dropout(ffn_output)) # shape: [B, S, d_model]

        return x


class Encoder(nn.Module):
    """
    Full encoder: N stacked encoder layers.

    The input flows through N=6 identical (but separately parameterized)
    encoder layers. Each layer refines the contextual representations.

    After N layers, the final output is a rich contextual representation
    of each source token, informed by all other source tokens.

    This output (sometimes called "memory" or "encoder_output") is passed
    to every decoder layer as the source of keys and values for cross-attention.
    """

    def __init__(self, n_layers: int = cfg.n_layers, d_model: int = cfg.d_model,
                 n_heads: int = cfg.n_heads, d_ff: int = cfg.d_ff,
                 dropout: float = cfg.dropout):
        super().__init__()

        # Create N identical-but-independent encoder layers.
        # nn.ModuleList is used (not a Python list) so PyTorch registers the sub-modules and includes their parameters in model.parameters().
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        """
        Pass input through all N encoder layers.

        Parameters
        ----------
        x : torch.Tensor, shape [B, S, d_model], Input after embedding + positional encoding.
        mask : torch.Tensor or None, shape [B, 1, 1, S], Source padding mask.

        Returns
        -------
        x : torch.Tensor, shape [B, S, d_model]
            Final encoder output (contextual representations of all source tokens).
            This is passed to each decoder layer as 'memory'.
        """
        # Pass through each encoder layer sequentially
        for layer in self.layers:
            x = layer(x, mask) # shape stays [B, S, d_model] through all layers

        return x

Overwriting encoder.py


# **7. Decoder Layer and Full Decoder Stack**

In [ ]:
%%writefile decoder.py
"""
decoder.py - Decoder Layer and Full Decoder Stack.

THE DECODER'S JOB
-----------------
The decoder generates the target sequence (e.g., a French translation) one
token at a time, conditioned on:
  1. The encoder output (source context - what we're translating FROM)
  2. Previously generated tokens (target context - what we've generated SO FAR)

During training, all target tokens are provided simultaneously (teacher
forcing), with the causal mask preventing future peeking.

During inference, tokens are generated autoregressively: the decoder is
called once per output token.

DECODER LAYER STRUCTURE
-----------------------
Each of the N=6 decoder layers applies, in order:

  1. Masked Multi-Head Self-Attention (causal):
       Each target position attends to all PREVIOUS target positions.
       Q = K = V = target embeddings (with causal + padding mask)

  2. Residual + LayerNorm

  3. Multi-Head Cross-Attention (encoder-decoder attention):
       Each target position attends to ALL source positions.
       Q = output from sub-layer 1 (decoder state)
       K = V = encoder_output (source representations), (with source padding mask)

  4. Residual + LayerNorm

  5. Position-wise FFN

  6. Residual + LayerNorm

KEY DIFFERENCE FROM ENCODER:
  - Has 3 sub-layers (encoder has 2)
  - Sub-layer 1 uses CAUSAL masking (encoder self-attention has none)
  - Sub-layer 2 is CROSS-attention (encoder only has self-attention)
  - The encoder output flows into EVERY decoder layer (not just the first)
"""

import torch
import torch.nn as nn
from attention import MultiHeadAttention
from feedforward import PositionwiseFeedForward
from config import cfg


class DecoderLayer(nn.Module):
    """
    A single decoder layer with 3 sub-layers:
      1. Masked self-attention
      2. Cross-attention (encoder-decoder)
      3. FFN

    All with residual connections and layer normalization.
    """

    def __init__(self, d_model: int = cfg.d_model, n_heads: int = cfg.n_heads,
                 d_ff: int = cfg.d_ff, dropout: float = cfg.dropout):
        super().__init__()

        # === Sub-layer 1: Masked Multi-Head Self-Attention ===
        # Operates on the target sequence. Uses causal masking: position i can only attend to positions 0..i.
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)

        # === Sub-layer 2: Multi-Head Cross-Attention (Encoder-Decoder) ===
        # Q comes from the decoder (previous sub-layer output). K and V come from the encoder output.
        # Allows each target position to look at all source positions.
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)

        # === Sub-layer 3: Position-wise FFN ===
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout)

        # === Layer Normalization (one per sub-layer) ===
        self.norm1 = nn.LayerNorm(d_model)  # after masked self-attention
        self.norm2 = nn.LayerNorm(d_model)  # after cross-attention
        self.norm3 = nn.LayerNorm(d_model)  # after FFN

        # === Dropout ===
        self.dropout = nn.Dropout(p=dropout)

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        memory_mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Forward pass through one decoder layer.

        Parameters
        ----------
        tgt : torch.Tensor, shape [B, S_tgt, d_model]
            Target sequence representations (after embedding + PE for first layer, or output of previous decoder layer for subsequent layers).

        memory : torch.Tensor, shape [B, S_src, d_model]
            Encoder output. Same for all decoder layers (passed in every time). This is what allows the decoder to "look at" the source sequence.

        tgt_mask : torch.Tensor or None, shape [B, 1, S_tgt, S_tgt]
            Combined causal + padding mask for the target sequence. True = block. Prevents attending to future or padding positions.

        memory_mask : torch.Tensor or None, shape [B, 1, 1, S_src]
            Padding mask for the source sequence. Prevents attending to padding tokens in the encoder output.

        Returns
        -------
        tgt : torch.Tensor, shape [B, S_tgt, d_model]
            Updated target representations.

        Complete Shape Trace:
        ---------------------
        B=2, S_tgt=4, S_src=5, d_model=512, n_heads=8, d_k=64
        tgt input:   [2, 4, 512]
        memory:      [2, 5, 512]  (from encoder)

        === Sub-layer 1: Masked Self-Attention ===
        residual:           [2, 4, 512]
        Q=K=V=tgt -> MHSA -> [2, 4, 512]
          (Q: [2,8,4,64], K: [2,8,4,64], V: [2,8,4,64])
          (scores: [2,8,4,4], masked by tgt_mask)
          (output: [2,8,4,64] -> combined -> [2,4,512])
        dropout:            [2, 4, 512]
        residual + out:     [2, 4, 512]
        norm1:              [2, 4, 512]

        === Sub-layer 2: Cross-Attention ===
        residual:                [2, 4, 512]
        Q=tgt, K=V=memory -> MHA:
          (Q: [2,8,4,64])        <- from decoder
          (K: [2,8,5,64])        <- from encoder!
          (V: [2,8,5,64])        <- from encoder!
          (scores: [2,8,4,5] masked by memory_mask)   <- 4 target queries attending to 5 source keys!
          (output: [2,8,4,64] -> combined -> [2,4,512])
        dropout:                 [2, 4, 512]
        residual + out:          [2, 4, 512]
        norm2:                   [2, 4, 512]

        === Sub-layer 3: FFN ===
        residual:           [2, 4, 512]
        FFN:                [2, 4, 512]
        dropout:            [2, 4, 512]
        residual + out:     [2, 4, 512]
        norm3:              [2, 4, 512]

        Output:             [2, 4, 512]  <- same shape as tgt input
        """

        # === Sub-layer 1: Masked Multi-Head Self-Attention ===
        residual = tgt
        # Self-attention: Q = K = V = tgt, tgt_mask prevents attending to future target positions (causal) and padding positions
        self_attn_out = self.self_attention(tgt, tgt, tgt, tgt_mask)
        tgt = self.norm1(residual + self.dropout(self_attn_out)) # shape: [B, S_tgt, d_model]

        # === Sub-layer 2: Multi-Head Cross-Attention ===
        residual = tgt
        # Cross-attention:
        #   Q = tgt (decoder state - "what the decoder is looking for")
        #   K = memory (encoder output - "what each source position represents")
        #   V = memory (encoder output - "what to retrieve from each source position")
        # memory_mask prevents attending to padding in the source
        cross_attn_out = self.cross_attention(tgt, memory, memory, memory_mask)
        tgt = self.norm2(residual + self.dropout(cross_attn_out)) # shape: [B, S_tgt, d_model]

        # === Sub-layer 3: Position-wise FFN ===
        residual = tgt
        ffn_out = self.feed_forward(tgt)
        tgt = self.norm3(residual + self.dropout(ffn_out)) # shape: [B, S_tgt, d_model]

        return tgt


class Decoder(nn.Module):
    """
    Full decoder: N stacked decoder layers.

    The target sequence flows through N=6 decoder layers.
    Each layer refines the target representations using:
      - Causal self-attention on the target
      - Cross-attention to the encoder output (passed in to EVERY layer)

    The SAME encoder output (memory) is passed to all decoder layers.
    """

    def __init__(self, n_layers: int = cfg.n_layers, d_model: int = cfg.d_model,
                 n_heads: int = cfg.n_heads, d_ff: int = cfg.d_ff,
                 dropout: float = cfg.dropout):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

    def forward(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        memory_mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Pass target through all N decoder layers.

        Parameters
        ----------
        tgt : torch.Tensor, shape [B, S_tgt, d_model]
        memory : torch.Tensor, shape [B, S_src, d_model], Encoder output (same for all decoder layers).
        tgt_mask : torch.Tensor or None, Target combined (causal + padding) mask.
        memory_mask : torch.Tensor or None, Source padding mask.

        Returns
        -------
        tgt : torch.Tensor, shape [B, S_tgt, d_model]
        """
        for layer in self.layers:
            tgt = layer(tgt, memory, tgt_mask, memory_mask) # shape stays [B, S_tgt, d_model] through all layers

        return tgt

Overwriting decoder.py


# **8. The complete Transformer model**

In [ ]:
%%writefile transformer.py
"""
transformer.py - The complete Transformer model.

This ties together all components:
  - Source embedding (token + positional)
  - Target embedding (token + positional)
  - Encoder stack
  - Decoder stack
  - Final linear + softmax projection

WEIGHT SHARING:
The paper shares weights between:
  1. Source token embedding matrix
  2. Target token embedding matrix
  3. The pre-softmax linear projection (output layer)

All three use the SAME weight matrix E of shape [vocab_size, d_model].
For the output linear layer, we use E^T: [d_model, vocab_size].

WHY SHARE WEIGHTS?
  - Reduces parameters significantly.
  - The embedding maps token_id -> vector; the output layer maps vector -> token_id.
    These are inverse operations - sharing weights makes sense intuitively.

PARAMETER INITIALIZATION:
  - Linear layers: Xavier/Glorot uniform (PyTorch default for nn.Linear)
  - Embeddings: N(0, 1) then scaled (we do this implicitly)
  - LayerNorm: gamma=1, beta=0 (PyTorch default)

We use Xavier uniform for all linear layers (this is PyTorch's default).
"""

import torch
import torch.nn as nn
import math
from embeddings import TransformerEmbedding, TokenEmbedding
from encoder import Encoder
from decoder import Decoder
from masks import make_src_mask, make_tgt_mask
from config import cfg


class Transformer(nn.Module):
    """
    The complete Transformer model for sequence-to-sequence tasks.

    FULL FORWARD PASS:
    ------------------
    1. Source tokens -> src_embedding (TokenEmb + PosEnc) -> [B, S_src, d_model]
    2. Encoder(src_emb, src_mask) -> encoder_output [B, S_src, d_model]
    3. Target tokens -> tgt_embedding (TokenEmb + PosEnc) -> [B, S_tgt, d_model]
    4. Decoder(tgt_emb, encoder_output, tgt_mask, src_mask) -> [B, S_tgt, d_model]
    5. output_projection(decoder_out) -> logits [B, S_tgt, vocab_size]

    During training: all steps happen in one forward() call.
    During inference: encode() once, then decode() autoregressively.
    """

    def __init__(
        self,
        vocab_size: int = cfg.vocab_size,
        d_model: int = cfg.d_model,
        n_heads: int = cfg.n_heads,
        n_layers: int = cfg.n_layers,
        d_ff: int = cfg.d_ff,
        dropout: float = cfg.dropout,
        max_seq_len: int = cfg.max_seq_len,
        pad_idx: int = cfg.PAD_IDX
    ):
        super().__init__()

        self.pad_idx = pad_idx
        self.d_model = d_model

        # Embeddings
        # We create ONE shared TokenEmbedding (the weight matrix E). Both src and tgt use it (weight sharing).
        # Each gets its own PositionalEncoding (PE is not shared - different sequences can have different lengths, and PE is fixed anyway).
        self.shared_embedding = TokenEmbedding(vocab_size, d_model)

        self.src_embedding = TransformerEmbedding(vocab_size, d_model, max_seq_len, dropout)
        self.tgt_embedding = TransformerEmbedding(vocab_size, d_model, max_seq_len, dropout)

        # Share weights: src and tgt token embeddings point to the SAME matrix
        self.src_embedding.token_embedding = self.shared_embedding
        self.tgt_embedding.token_embedding = self.shared_embedding

        # Encoder
        self.encoder = Encoder(n_layers, d_model, n_heads, d_ff, dropout)

        # Decoder
        self.decoder = Decoder(n_layers, d_model, n_heads, d_ff, dropout)

        # Output projection: d_model -> vocab_size. This is a linear layer WITHOUT bias. Weight matrix shape: [vocab_size, d_model]
        # We transpose to apply as [d_model, vocab_size] in the forward pass.
        self.output_projection = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: output_projection weight = shared embedding weight
        # output_projection.weight has shape [vocab_size, d_model], shared_embedding.embedding.weight has shape [vocab_size, d_model]
        # They are the SAME tensor - not a copy!
        self.output_projection.weight = self.shared_embedding.embedding.weight

        # Initialize parameters
        self._init_parameters()

    def _init_parameters(self):
        """
        Initialize model parameters.

        For linear layers: Xavier uniform initialization.
        For layer norms: already initialized to gamma=1, beta=0 by PyTorch.
        For embeddings: PyTorch default N(0,1).
        For biases: initialized to zero.
        """
        for name, p in self.named_parameters():
            if p.dim() > 1:
                # 2D+ tensors: weight matrices of linear layers
                nn.init.xavier_uniform_(p)
            elif 'bias' in name:
                nn.init.zeros_(p)
            # LayerNorm params (gamma, beta) and embedding weights keep PyTorch defaults

    def make_masks(self, src: torch.Tensor, tgt: torch.Tensor):
        """
        Create all masks needed for a forward pass.

        Parameters
        ----------
        src : [B, S_src]  — source token IDs
        tgt : [B, S_tgt]  — target token IDs

        Returns
        -------
        src_mask    : [B, 1, 1, S_src]      - padding mask for source
        tgt_mask    : [B, 1, S_tgt, S_tgt]  - causal + padding for target
        """
        src_mask = make_src_mask(src, self.pad_idx)
        tgt_mask = make_tgt_mask(tgt, self.pad_idx)
        return src_mask, tgt_mask

    def encode(self, src: torch.Tensor, src_mask: torch.Tensor = None) -> torch.Tensor:
        """
        Run the encoder on the source sequence.
        Called once per inference pass (not per generated token).

        Parameters
        ----------
        src : [B, S_src]            - source token IDs
        src_mask : [B, 1, 1, S_src] - source padding mask

        Returns
        -------
        memory : [B, S_src, d_model] - encoder output
        """
        if src_mask is None:
            src_mask = make_src_mask(src, self.pad_idx)

        src_emb = self.src_embedding(src)       # [B, S_src, d_model]
        memory = self.encoder(src_emb, src_mask) # [B, S_src, d_model]
        return memory

    def decode(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        memory_mask: torch.Tensor = None
    ) -> torch.Tensor:
        """
        Run the decoder on target sequence + encoder memory.

        Parameters
        ----------
        tgt : [B, S_tgt]             - target token IDs so far
        memory : [B, S_src, d_model]  - encoder output
        tgt_mask : [B, 1, S_tgt, S_tgt]
        memory_mask : [B, 1, 1, S_src]

        Returns
        -------
        logits : [B, S_tgt, vocab_size] - raw scores for each vocab token
        """
        if tgt_mask is None:
            tgt_mask = make_tgt_mask(tgt, self.pad_idx)

        tgt_emb = self.tgt_embedding(tgt)   # [B, S_tgt, d_model]
        dec_out = self.decoder(tgt_emb, memory, tgt_mask, memory_mask)   # [B, S_tgt, d_model]

        logits = self.output_projection(dec_out)   # [B, S_tgt, vocab_size]

        return logits

    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor
    ) -> torch.Tensor:
        """
        Full forward pass: src + tgt -> logits.

        Used during TRAINING (teacher forcing: entire target sequence fed at once).

        Parameters
        ----------
        src : shape [B, S_src], Source token IDs.
        tgt : shape [B, S_tgt] Target input token IDs (decoder input - shifted right, starts with <SOS>).
            In teacher forcing, this is ground-truth target[:-1] with <SOS> prepended.

        Returns
        -------
        logits : shape [B, S_tgt, vocab_size]
            Raw (unnormalized) scores for each vocabulary token at each position.
            Apply softmax to get probabilities; use with cross-entropy loss directly.

        Full Shape Trace (B=2, S_src=5, S_tgt=4, d_model=512, vocab_size=1000):
        src:            [2, 5]
        tgt:            [2, 4]

        src_mask:       [2, 1, 1, 5]
        tgt_mask:       [2, 1, 4, 4]

        src_emb:        [2, 5, 512]
        encoder_out:    [2, 5, 512]

        tgt_emb:        [2, 4, 512]
        decoder_out:    [2, 4, 512]

        logits:         [2, 4, 1000]
        """
        # Build masks
        src_mask, tgt_mask = self.make_masks(src, tgt)

        # Encode source
        memory = self.encode(src, src_mask)       # [B, S_src, d_model]

        # Decode target (with cross-attention to memory)
        logits = self.decode(tgt, memory, tgt_mask, src_mask)   # [B, S_tgt, vocab_size]

        return logits

    def count_parameters(self) -> int:
        """Count total trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def build_transformer(config=cfg) -> Transformer:
    """
    Factory function: build a Transformer from config.
    Prints a summary of the model.
    """
    model = Transformer(
        vocab_size=config.vocab_size,
        d_model=config.d_model,
        n_heads=config.n_heads,
        n_layers=config.n_layers,
        d_ff=config.d_ff,
        dropout=config.dropout,
        max_seq_len=config.max_seq_len,
        pad_idx=config.PAD_IDX,
    )
    n_params = model.count_parameters()
    print(f"[Transformer] Built model with {n_params:,} trainable parameters")
    print(f"  d_model={config.d_model}, n_heads={config.n_heads}, "
          f"n_layers={config.n_layers}, d_ff={config.d_ff}")
    return model

Overwriting transformer.py


# **9. Label Smoothing Cross-Entropy Loss**

In [ ]:
%%writefile loss.py
"""
loss.py - Label Smoothing Cross-Entropy Loss.
WHAT LABEL SMOOTHING IS
-----------------------
Standard cross-entropy loss trains the model against one-hot targets: y = [0, 0, 1, 0, 0, ...]   (1.0 at the correct token, 0 elsewhere)
The model is pushed to put ALL probability mass on the correct token:
  predicted prob of correct token -> 1.0
  predicted prob of all others    -> 0.0

This causes two problems:
  1. OVERCONFIDENCE: The model outputs extreme logit values to satisfy the loss, which uses model capacity inefficiently.
  2. POOR CALIBRATION: The model assigns ~0 probability to alternatives, even sensible ones (many translations are valid!).

Label smoothing distributes a small amount e-epsilon of probability mass uniformly across ALL vocabulary tokens, including the correct one:

  y_smooth = (1 - e) * y_onehot + e / V

With e=0.1 and V=1000:
  - Correct token target: 0.9 + 0.1/1000 = 0.9001
  - Each wrong token target: 0.1/1000 = 0.0001

The model is now trained to be "almost" certain (not fully certain) about the correct token. This is a form of regularization.

EFFECT ON LOSS:
  L = -sum_i y_smooth_i * log(p_i) = -(1-e) * log(p_correct) - e/V * sum_i log(p_i)

The second term penalizes the model for having a very peaked (confident) distribution - it receives a lower loss if it spreads some probability.

THE PAPER'S RESULT:
"Label smoothing of value e_ls = 0.1, hurts perplexity, as the model learns to be more unsure, but improves accuracy and BLEU score."

Perplexity measures confidence -> lower perplexity = more confident. Smoothing makes the model less confident -> higher perplexity.
But it generalizes better -> better BLEU.

PADDING:
We ignore PAD tokens in the loss (they have no meaningful target). The loss is averaged only over non-PAD positions.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from config import cfg


class LabelSmoothingLoss(nn.Module):
    """
    Cross-entropy loss with label smoothing and PAD token masking.

    Can be used in two modes:
      1. With PyTorch's built-in label_smoothing (simpler, recommended)
      2. Manual KL-divergence formulation (more explicit, educational)

    We implement both and use PyTorch's built-in for training.
    """

    def __init__(
        self,
        vocab_size: int = cfg.vocab_size,
        pad_idx: int = cfg.PAD_IDX,
        label_smoothing: float = cfg.label_smoothing
    ):
        """
        Parameters
        ----------
        vocab_size : int, Number of tokens in vocabulary (size of logit dimension).
        pad_idx : int, Token ID for padding. These positions are ignored in loss.
        label_smoothing : float, Epsilon e in [0, 1). 0 = no smoothing (standard CE). 0.1 = paper value.
        """
        super().__init__()
        self.vocab_size = vocab_size
        self.pad_idx = pad_idx
        self.label_smoothing = label_smoothing

        # PyTorch's CrossEntropyLoss with label_smoothing handles everything:
        # - Softmax is applied internally (expects raw logits, not probabilities)
        # - label_smoothing distributes e mass uniformly
        # - ignore_index masks PAD positions (they contribute 0 to the loss)
        # - reduction='mean' averages over all non-ignored positions
        self.criterion = nn.CrossEntropyLoss(
            label_smoothing=label_smoothing,
            ignore_index=pad_idx,
            reduction='mean'
        )

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Compute label-smoothed cross-entropy loss.

        Parameters
        ----------
        logits : shape [B, S, vocab_size], Raw (unnormalized) model outputs. DO NOT apply softmax before this.
            nn.CrossEntropyLoss applies log_softmax internally.
        targets : shape [B, S], Ground-truth token IDs. PAD positions are ignored.
            These should be the SHIFTED target: target[1:] (the tokens to predict).

        Returns
        -------
        loss : scalar, Scalar loss value averaged over all non-PAD positions.

        Shape trace:
          logits:  [B, S, vocab_size]  e.g., [2, 4, 1000]
          targets: [B, S]              e.g., [2, 4]

          nn.CrossEntropyLoss expects:
            input:  [N, C]
            target: [N]
          where C is number of classes.

          We reshape:
            logits:  [B*S, vocab_size]  <- [N, C] format
            targets: [B*S]              <- [N] format

          loss: scalar (mean over non-PAD positions)
        """
        B, S, V = logits.shape

        # Reshape for CrossEntropyLoss:
        logits_flat = logits.reshape(B * S, V)

        # [B, S] -> [B*S]
        targets_flat = targets.reshape(B * S)

        loss = self.criterion(logits_flat, targets_flat)
        return loss


def compute_loss_manual(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pad_idx: int = cfg.PAD_IDX,
    label_smoothing: float = cfg.label_smoothing
) -> torch.Tensor:
    """
    Manual implementation of label-smoothed cross-entropy for understanding.
    The LabelSmoothingLoss class above uses PyTorch's built-in, which is more efficient.

    DERIVATION:
    -----------
    Standard CE:    L = -log(p_{y})
    KL divergence:  L = sum_i y_smooth_i * log(y_smooth_i / p_i)
                      = sum_i y_smooth_i * log(y_smooth_i)  (constant) - sum_i y_smooth_i * log(p_i)  (cross-entropy)

    Ignoring the constant (doesn't affect gradients):
      L = -sum_i y_smooth_i * log(p_i) = -(1-e) * log(p_y) - (e/V) * sum_i log(p_i)
    sum_i log(p_i) = sum of log-probs over all tokens.
    """
    B, S, V = logits.shape

    # Log probabilities: [B, S, vocab_size]
    log_probs = F.log_softmax(logits, dim=-1)

    # Gather log prob of the correct token at each position: [B, S]
    # targets.unsqueeze(-1): [B, S, 1]
    # log_probs.gather(-1, ...): [B, S, 1] -> [B, S]
    log_prob_correct = log_probs.gather(-1, targets.unsqueeze(-1)).squeeze(-1)

    # Sum of log probs over all vocab: [B, S]
    log_prob_sum = log_probs.sum(dim=-1)

    # Label-smoothed loss per position:
    # L = -(1-e) * log(p_y) - (e/V) * sum_i log(p_i)
    loss_per_pos = -(1 - label_smoothing) * log_prob_correct \
                   - (label_smoothing / V) * log_prob_sum

    # Mask: 1 for real tokens, 0 for PAD
    non_pad_mask = (targets != pad_idx).float()  # [B, S]

    # Average over non-PAD positions
    loss = (loss_per_pos * non_pad_mask).sum() / non_pad_mask.sum().clamp(min=1)
    return loss

Overwriting loss.py


# **10. Adam optimizer with Noam (Transformer) learning rate schedule**

In [ ]:
%%writefile optimizer.py
"""
optimizer.py — Adam optimizer with Noam (Transformer) learning rate schedule.

THE SCHEDULE FROM THE PAPER (Section 5.3):
-----------------------------------------
lrate = d_model^{-0.5} * min(step_num^{-0.5}, step_num * warmup_steps^{-1.5})

This schedule has two phases:
  PHASE 1 (step ≤ warmup_steps): LINEAR WARMUP
    The active term is: step_num * warmup_steps^{-1.5}
    This grows linearly from ~0 to peak. Peak is at step = warmup_steps.

  PHASE 2 (step > warmup_steps): INVERSE SQRT DECAY
    The active term is: step_num^{-0.5}. This decays as 1/sqrt(step), slowly decreasing.

Warmup gives the optimizer time to:
  1. Build up reliable gradient statistics
  2. Allow the model to find a reasonable initial direction before large steps

WHY INVERSE SQRT DECAY?
-----------------------
As training progresses, the model gets closer to a good solution. We want smaller updates to "fine-tune" rather than overshoot. 1/sqrt(step) is
a common schedule - it's slow enough to keep making progress, but decreasing enough to converge.

OPTIMIZER HYPERPARAMETERS:
  beta_1 = 0.9   (momentum: exponential decay of past gradients)
  beta_2 = 0.98  (RMSProp: exponential decay of past squared gradients)
  e = 1e-9    (numerical stability; very small to let Adam's learning rate schedule dominate)

"""

import torch
import torch.optim as optim
from config import cfg


class NoamScheduler:
    """
    The Noam learning rate scheduler from "Attention Is All You Need."

    This is NOT a PyTorch LRScheduler subclass - it manually sets the
    learning rate at each step. This is simpler and more transparent.

    Usage:
        optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
        scheduler = NoamScheduler(optimizer, d_model=512, warmup_steps=4000)

        for batch in data:
            loss = ...
            loss.backward()
            optimizer.step()
            scheduler.step()   <- call AFTER optimizer.step()
            optimizer.zero_grad()
    """

    def __init__(
        self,
        optimizer: optim.Optimizer,
        d_model: int = cfg.d_model,
        warmup_steps: int = cfg.warmup_steps,
        factor: float = 1.0
    ):
        """
        Parameters
        ----------
        optimizer : torch.optim.Optimizer, The optimizer whose learning rate we modify.
            Should be initialized with lr=1.0 (we'll override it every step).
        d_model : int, Model dimension. Appears in the formula as d_model^{-0.5}.
        warmup_steps : int, Number of warmup steps. Paper: 4000.
        factor : float Scaling factor.
        """
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.factor = factor
        self._step = 0      # current step number (0-indexed, incremented before use)
        self._rate = 0.0    # current learning rate (for logging)

    def step(self):
        """
        Increment step counter and update learning rate. Call this AFTER optimizer.step() each training iteration.
        """
        self._step += 1
        rate = self._compute_lr(self._step)
        self._rate = rate

        # Set the learning rate in all parameter groups of the optimizer
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = rate

    def _compute_lr(self, step: int) -> float:
        """
        Compute the learning rate for a given step.
        Formula:
          lr = factor * d_model^{-0.5} * min(step^{-0.5}, step * warmup^{-1.5})

        Parameters
        ----------
        step : int, Current training step (1-indexed).

        Returns
        -------
        lr : float, Learning rate for this step.
        """
        # Avoid step=0 (division by zero / undefined)
        step = max(step, 1)

        # d_model^{-0.5}
        d_model_factor = self.d_model ** (-0.5)

        # min(step^{-0.5}, step * warmup^{-1.5})
        arg1 = step ** (-0.5)                         # decay term
        arg2 = step * (self.warmup_steps ** (-1.5))   # warmup term

        return self.factor * d_model_factor * min(arg1, arg2)

    @property
    def current_lr(self) -> float:
        """Return the current learning rate."""
        return self._rate

    @property
    def current_step(self) -> int:
        """Return the current step number."""
        return self._step

    def state_dict(self) -> dict:
        """Save scheduler state for checkpointing."""
        return {
            'd_model': self.d_model,
            'warmup_steps': self.warmup_steps,
            'factor': self.factor,
            '_step': self._step,
            '_rate': self._rate,
        }

    def load_state_dict(self, state: dict):
        """Restore scheduler state from checkpoint."""
        self._step = state['_step']
        self._rate = state['_rate']
        # Apply the loaded LR to optimizer immediately
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = self._rate


def build_optimizer(model: torch.nn.Module, config=cfg):
    """
    Build the Adam optimizer and Noam LR scheduler from config.

    Returns
    -------
    optimizer : torch.optim.Adam
    scheduler : NoamScheduler
    """
    # Adam with paper's hyperparameters. # lr=1.0 is a placeholder - NoamScheduler overwrites it every step.
    optimizer = optim.Adam(
        model.parameters(),
        lr=0.0,         # overridden by scheduler
        betas=(0.9, 0.98),
        eps=1e-9
    )

    scheduler = NoamScheduler(
        optimizer,
        d_model=config.d_model,
        warmup_steps=config.warmup_steps,
        factor=0.5        # changed from 1.0 to 0.5
    )

    return optimizer, scheduler

Writing optimizer.py


# **11. Sequence-to-Sequence Dataset**

In [ ]:
%%writefile dataset.py
"""
dataset.py - Sequence-to-Sequence Translation Dataset.

THE TASK: ENGLISH TO GERMAN TRANSLATION
---------------------------------------
This pipeline processes real-world bilingual data. It handles:
  1. Disk I/O (loading from JSONL files)
  2. Subword Tokenization (via trained HuggingFace BPE tokenizer)
  3. Truncation (enforcing cfg.max_seq_len)
  4. Dynamic Padding (in the collate_fn)

SEQUENCE FORMAT
---------------
Source input   (src):     [t1, t2, t3, ...] (Truncated to max_seq_len)
Decoder input  (tgt_in):  [<SOS>, t1, t2, t3, ...]
Decoder target (tgt_out): [t1, t2, t3, ..., <EOS>]

TRUNCATION LOGIC
----------------
Because sequences can be arbitrarily long, we must truncate them to fit our GPU memory budget (max_seq_len = 128).
For the target sequences, we truncate to max_seq_len - 1 to guarantee we always have room to append the <SOS> or <EOS> token without exceeding 128.
"""

import json
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from typing import List, Tuple
from tokenizers import Tokenizer
from config import cfg

class TranslationDataset(Dataset):
    """
    PyTorch Dataset for English-to-German Translation.

    Reads a JSONL file where each line is:
    {"translation": {"en": "Hello", "de": "Hallo"}}
    """

    def __init__(
        self,
        jsonl_path: str,
        tokenizer_path: str = "bpe_tokenizer.json",
        max_len: int = cfg.max_seq_len
    ):
        """
        Parameters
        ----------
        jsonl_path : str
            Path to the train.jsonl, val.jsonl, or test.jsonl file.
        tokenizer_path : str
            Path to the saved BPE tokenizer JSON file.
        max_len : int
            Maximum absolute sequence length (from config).
        """
        self.max_len = max_len

        # 1. Load the Tokenizer
        self.tokenizer = Tokenizer.from_file(tokenizer_path)

        # 2. Map Special Tokens (must match tokenizer training order)
        # Training order: ["<pad>", "<sos>", "<eos>", "<unk>"]
        self.PAD = self.tokenizer.token_to_id("<pad>")
        self.SOS = self.tokenizer.token_to_id("<sos>")
        self.EOS = self.tokenizer.token_to_id("<eos>")
        self.UNK = self.tokenizer.token_to_id("<unk>")

        # 3. Load Data from Disk
        self.data = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                item = json.loads(line)
                self.data.append(item['translation'])

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Returns one processed sample.
        """
        pair = self.data[idx]
        en_text = pair['en']
        de_text = pair['de']

        # 1. Encode text to integer IDs
        src_ids = self.tokenizer.encode(en_text).ids
        tgt_ids = self.tokenizer.encode(de_text).ids

        # 2. Truncate
        # Source can be truncated exactly to max_len
        src_ids = src_ids[:self.max_len]

        # Target must be truncated to max_len - 1 to leave room for SOS/EOS
        tgt_ids = tgt_ids[:self.max_len - 1]

        # 3. Convert to Tensors and apply offsets
        src = torch.tensor(src_ids, dtype=torch.long)

        # tgt_in: prepend SOS, no EOS
        tgt_in = torch.tensor([self.SOS] + tgt_ids, dtype=torch.long)

        # tgt_out: no SOS, append EOS
        tgt_out = torch.tensor(tgt_ids + [self.EOS], dtype=torch.long)

        return src, tgt_in, tgt_out


def collate_fn(
    batch: List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]],
    pad_idx: int = 0  # <pad> is ID 0
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Collate a batch of variable-length sequences into padded tensors.
    Uses Dynamic Padding.
    """
    srcs, tgt_ins, tgt_outs = zip(*batch)

    # Pad sequences to the longest item in THIS specific batch
    src_batch = pad_sequence(srcs, batch_first=True, padding_value=pad_idx)
    tgt_in_batch = pad_sequence(tgt_ins, batch_first=True, padding_value=pad_idx)
    tgt_out_batch = pad_sequence(tgt_outs, batch_first=True, padding_value=pad_idx)

    return src_batch, tgt_in_batch, tgt_out_batch


def build_dataloaders(
    train_path: str,
    val_path: str,
    tokenizer_path: str,
    batch_size: int = cfg.batch_size,
    num_workers: int = 2
) -> Tuple[DataLoader, DataLoader]:
    """
    Build training and validation DataLoaders.
    """
    train_dataset = TranslationDataset(train_path, tokenizer_path)
    val_dataset = TranslationDataset(val_path, tokenizer_path)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,          # Shuffle training data
        collate_fn=lambda b: collate_fn(b, pad_idx=train_dataset.PAD),
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,         # No need to shuffle validation data
        collate_fn=lambda b: collate_fn(b, pad_idx=val_dataset.PAD),
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available()
    )

    print(f"[Dataset] Train: {len(train_dataset):,} samples | "
          f"Val: {len(val_dataset):,} samples | "
          f"Batch size: {batch_size}")

    return train_loader, val_loader

Overwriting dataset.py


# **12. Full Training Loop for the Transformer**

In [ ]:
%%writefile train.py
"""
train.py - Full Training Loop for the Transformer.

WHAT THIS FILE DOES
-------------------
Ties together:
  - Dataset / DataLoaders
  - Model (Transformer)
  - Loss function (LabelSmoothingLoss)
  - Optimizer (Adam)
  - LR Scheduler (NoamScheduler)
  - Training loop (forward, loss, backward, step)
  - Validation loop
  - Checkpointing (save/load best model)
  - Logging (loss, LR, accuracy per epoch)

THE TRAINING LOOP IN DETAIL
---------------------------
For each epoch:
  1. Set model to train mode (enables dropout)
  2. For each batch:
     a. Move src, tgt_in, tgt_out to device
     b. Forward pass: logits = model(src, tgt_in)
     c. Compute loss: loss = criterion(logits, tgt_out)
     d. Backward pass: loss.backward()  (computes gradients)
     e. Gradient clipping: clip_grad_norm_(model.parameters(), max_norm=1.0)
     f. Optimizer step: optimizer.step()  (updates parameters)
     g. LR scheduler step: scheduler.step()
     h. Zero gradients: optimizer.zero_grad()
  3. Compute training metrics (loss, token accuracy)
  4. Run validation loop (same as training but no backward/step)
  5. Save checkpoint if validation loss improved
  6. Print epoch summary

TEACHER FORCING:
During training, tgt_in = [SOS, t1, t2, ..., t_{n-1}] (ground truth shifted right).
The model predicts tgt_out = [t1, t2, ..., t_n, EOS].
All positions are computed in parallel (causal mask prevents cheating).

TOKEN ACCURACY:
We measure how many output tokens (excluding PAD) the model correctly predicts.
This is a proxy for sequence accuracy - a sequence is only "correct" if ALL
tokens are correct, but token accuracy is easier to compute and more informative
during early training.

CHECKPOINTING:
We save:
  - model.state_dict(): all parameters
  - optimizer.state_dict(): Adam moment estimates (allows resuming training)
  - scheduler.state_dict(): current step (for LR schedule continuity)
  - epoch, best_val_loss: metadata
"""

import os
import time
import math
import torch
import torch.nn as nn
from typing import Tuple, Dict, Optional

from config import cfg
from transformer import build_transformer, Transformer
from loss import LabelSmoothingLoss
from optimizer import build_optimizer, NoamScheduler
from dataset import build_dataloaders


def compute_token_accuracy(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pad_idx: int = cfg.PAD_IDX
) -> Tuple[int, int]:
    """
    Compute number of correctly predicted tokens (ignoring PAD).

    Parameters
    ----------
    logits : [B, S, vocab_size]
    targets : [B, S]
    pad_idx : int

    Returns
    -------
    correct : int   — number of correct non-PAD predictions
    total : int     — total number of non-PAD tokens
    """
    # predicted: [B, S] - argmax over vocab dimension
    predicted = logits.argmax(dim=-1)

    # Create mask for non-PAD positions
    non_pad = (targets != pad_idx)

    # Count correct predictions at non-PAD positions
    correct = ((predicted == targets) & non_pad).sum().item()
    total = non_pad.sum().item()

    return correct, total


def train_epoch(
    model: Transformer,
    dataloader,
    criterion: LabelSmoothingLoss,
    optimizer: torch.optim.Optimizer,
    scheduler: NoamScheduler,
    device: torch.device,
    clip_grad: float = 1.0
) -> Dict[str, float]:
    """
    Run one training epoch.
    Returns
    -------
    metrics : dict with keys 'loss', 'accuracy', 'lr'
    """
    model.train()   # Enable dropout

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    num_batches = 0

    for batch_idx, (src, tgt_in, tgt_out) in enumerate(dataloader):
        # Move to device
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)

        # Forward pass
        # model(src, tgt_in) -> logits [B, S_tgt, vocab_size]
        # src:     [B, S_src]   — source token IDs
        # tgt_in:  [B, S_tgt]   — target input (SOS + target[:-1])
        # tgt_out: [B, S_tgt]   — target output (target[1:] + EOS)
        logits = model(src, tgt_in) # logits: [B, S_tgt, vocab_size]

        # Compute loss, criterion expects logits [B, S, V] and targets [B, S]
        loss = criterion(logits, tgt_out)

        # Backward pass
        loss.backward() # Gradients now accumulated in all parameter .grad attributes

        # Gradient clipping
        # Clip the global L2 norm of all gradients to `clip_grad`.
        # If norm > clip_grad, all gradients are scaled down proportionally.
        # This prevents exploding gradients from destabilizing training.
        if clip_grad > 0:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        # 1. Calculate and apply the EXACT learning rate for this step
        scheduler.step()

        # 2. Update the weights using that precise learning rate
        optimizer.step()

        # 3. Clear gradients
        optimizer.zero_grad(set_to_none=True)

        # Accumulate metrics
        with torch.no_grad():
            correct, total = compute_token_accuracy(logits, tgt_out, cfg.PAD_IDX)
            total_correct += correct
            total_tokens += total
            total_loss += loss.item()
            num_batches += 1

    return {
        'loss': total_loss / max(num_batches, 1),
        'accuracy': total_correct / max(total_tokens, 1),
        'lr': scheduler.current_lr
    }

@torch.no_grad()
def validate_epoch(
    model: Transformer,
    dataloader,
    criterion: LabelSmoothingLoss,
    device: torch.device
) -> Dict[str, float]:
    """
    Run one validation epoch (no gradient computation).

    Returns
    ───────
    metrics : dict with keys 'loss', 'accuracy'
    """
    model.eval()   # Disable dropout

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0
    num_batches = 0

    for src, tgt_in, tgt_out in dataloader:
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)

        logits = model(src, tgt_in)
        loss = criterion(logits, tgt_out)

        correct, total = compute_token_accuracy(logits, tgt_out, cfg.PAD_IDX)
        total_correct += correct
        total_tokens += total
        total_loss += loss.item()
        num_batches += 1

    return {
        'loss': total_loss / max(num_batches, 1),
        'accuracy': total_correct / max(total_tokens, 1)
    }

def save_checkpoint(
    model: Transformer,
    optimizer: torch.optim.Optimizer,
    scheduler: NoamScheduler,
    epoch: int,
    val_loss: float,
    history: dict,
    path: str
):
    """Save model checkpoint to disk."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': val_loss,
        'history': history
    }
    torch.save(checkpoint, path)
    print(f"  [Checkpoint] Saved to {path}")


def load_checkpoint(
    path: str,
    model: Transformer,
    optimizer: Optional[torch.optim.Optimizer] = None,
    scheduler: Optional[NoamScheduler] = None
) -> Tuple[int, dict, float]:
    """
    Load model (and optionally optimizer/scheduler) from checkpoint.

    Returns
    -------
    epoch : int — epoch number when checkpoint was saved
    """
    checkpoint = torch.load(path, map_location='cpu')
    model.load_state_dict(checkpoint['model_state_dict'])
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    epoch = checkpoint.get('epoch', 0)
    val_loss = checkpoint.get('val_loss', float('inf'))

    # Extract history, or return empty template if it doesn't exist
    empty_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    history = checkpoint.get('history', empty_history)

    print(f"  [Checkpoint] Loaded from {path} (epoch {epoch}, val_loss {val_loss:.4f})")
    return epoch, history, val_loss

def train(
    num_epochs: int = cfg.num_epochs,
    batch_size: int = cfg.batch_size,
    checkpoint_dir: str = './checkpoints',
    resume_from: str = None,
    task: str = 'translation',
    verbose: bool = True
):
    """
    Full training pipeline.

    Parameters
    ----------
    num_epochs : int
    batch_size : int
    checkpoint_dir : str - directory to save checkpoints
    resume_from : str or None - path to checkpoint to resume from
    task : str - Task identifier
    verbose : bool - print per-epoch stats
    """
    os.makedirs(checkpoint_dir, exist_ok=True)
    device = cfg.device
    print(f"\n{'='*60}")
    print(f"TRAINING TRANSFORMER")
    print(f"{'='*60}")
    print(f"  Device:   {device}")
    print(f"  Task:     {task}")
    print(f"  Epochs:   {num_epochs}")
    print(f"  Batch:    {batch_size}")

    # Build components
    model = build_transformer(cfg).to(device)
    criterion = LabelSmoothingLoss(
        vocab_size=cfg.vocab_size,
        pad_idx=cfg.PAD_IDX,
        label_smoothing=cfg.label_smoothing
    )
    optimizer, scheduler = build_optimizer(model, cfg)

    # Initialize State
    start_epoch = 0
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

    # Optionally resume from checkpoint
    if resume_from and os.path.exists(resume_from):
        start_epoch, history, current_val_loss = load_checkpoint(
            resume_from, model, optimizer, scheduler
        )

        # Find the TRUE historical best from the loaded history list!
        if len(history['val_loss']) > 0:
            best_val_loss = min(history['val_loss'])
        else:
            best_val_loss = current_val_loss

    # Build dataloaders
    train_loader, val_loader = build_dataloaders(
        train_path="data/train.jsonl",
        val_path="data/val.jsonl",
        tokenizer_path="bpe_tokenizer.json",
        batch_size=batch_size
    )

    # Training loop
    print(f"\n{'Epoch':>6}  {'Train Loss':>10}  {'Val Loss':>10}  "
          f"{'Train Acc':>10}  {'Val Acc':>10}  {'LR':>12}  {'Time':>8}")
    print("-" * 78)

    for epoch in range(start_epoch + 1, num_epochs + 1):
        t_start = time.time()

        # Train
        train_metrics = train_epoch(
            model, train_loader, criterion, optimizer, scheduler, device
        )

        # Validate
        val_metrics = validate_epoch(model, val_loader, criterion, device)

        # Log
        elapsed = time.time() - t_start
        history['train_loss'].append(train_metrics['loss'])
        history['val_loss'].append(val_metrics['loss'])
        history['train_acc'].append(train_metrics['accuracy'])
        history['val_acc'].append(val_metrics['accuracy'])
        history['lr'].append(train_metrics['lr'])

        if verbose:
            print(f"{epoch:>6}  {train_metrics['loss']:>10.4f}  {val_metrics['loss']:>10.4f}  "
                  f"{train_metrics['accuracy']:>10.2%}  {val_metrics['accuracy']:>10.2%}  "
                  f"{train_metrics['lr']:>12.2e}  {elapsed:>6.1f}s")

        # Checkpoint if best
        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            save_checkpoint(
                model, optimizer, scheduler, epoch, val_metrics['loss'], history,
                os.path.join(checkpoint_dir, 'best_model.pt')
            )

        # Save latest checkpoint every 5 epochs (for resuming)
        if epoch % 5 == 0:
            save_checkpoint(
                model, optimizer, scheduler, epoch, val_metrics['loss'], history,
                os.path.join(checkpoint_dir, f'epoch_{epoch:04d}.pt')
            )

    print(f"\n{'='*60}")
    print(f"Training complete. Best val loss: {best_val_loss:.4f}")
    print(f"{'='*60}")

    return model, history

Overwriting train.py


# **13. Autoregressive decoding at inference time**

In [ ]:
%%writefile inference.py
"""
inference.py - Autoregressive decoding at inference time.

TRAINING vs INFERENCE
---------------------
During TRAINING (teacher forcing):
  - We feed the ENTIRE target sequence to the decoder at once.
  - The causal mask prevents the decoder from seeing future tokens.
  - All output logits are computed in a single forward pass.
  - This is efficient because it's fully parallelized.

During INFERENCE:
  - We don't have the target sequence (that's what we want to generate!).
  - We must generate tokens one at a time, left to right.
  - At step t, we feed tokens [SOS, t1, ..., t_{t-1}] to get logit for t_t.
  - Then append t_t and repeat until <EOS> or max_len.
  - This IS sequential - no parallelism across generation steps.
  - But we can parallelize across multiple sequences in a batch.

GREEDY DECODING vs BEAM SEARCH
------------------------------
GREEDY: At each step, pick the token with the highest probability.
  - Simple and fast
  - Locally optimal at each step, but not globally optimal
  - Often sufficient for simple tasks

BEAM SEARCH: At each step, keep the top-k most likely SEQUENCES (beams).
  - Explores multiple hypotheses simultaneously
  - Much better for real translation (the paper uses beam size 4)
  - More complex to implement

We implement both: greedy for simplicity, beam search for completeness.
"""

import torch
import torch.nn.functional as F
from typing import List, Optional
from config import cfg


@torch.no_grad()
def greedy_decode(
    model: torch.nn.Module,
    src: torch.Tensor,
    max_len: int = cfg.max_seq_len,
    sos_idx: int = cfg.SOS_IDX,
    eos_idx: int = cfg.EOS_IDX,
    pad_idx: int = cfg.PAD_IDX,
    device: torch.device = None
) -> torch.Tensor:
    """
    Greedy autoregressive decoding: always pick the most probable next token.
    Parameters
    ----------
    model : Transformer, The trained model. Must be in eval mode (model.eval()).
    src : torch.Tensor, shape [B, S_src] or [S_src], Source token IDs. If 1D, adds batch dimension automatically.
    max_len : int, Maximum number of output tokens to generate (including EOS).
    sos_idx : int, Start-of-sequence token ID.
    eos_idx : int, End-of-sequence token ID. Generation stops when this is produced.
    pad_idx : int, Padding token ID (for the source mask).
    device : torch.device, Device to run on.

    Returns
    ------
    decoded : shape [B, S_out]
        Generated token IDs for each sequence in the batch. May have different lengths if we pad to the longest.
        Includes EOS (if generated), excludes SOS.

    STEP-BY-STEP:
    -------------
    1. Encode source once -> memory [B, S_src, d_model]
    2. Initialize decoder input = [[SOS], [SOS], ...] shape [B, 1]
    3. For t = 1, 2, ..., max_len:
       a. Run decoder(decoder_input, memory) -> logits [B, t, vocab_size]
       b. Take last position: logits[:, -1, :] -> [B, vocab_size]
       c. Argmax -> next_token [B]
       d. Append to decoder_input
       e. If ALL sequences have produced EOS, stop early
    4. Return decoder_input[:, 1:] (remove SOS prefix)
    """
    if device is None:
        device = next(model.parameters()).device

    # Ensure src has batch dimension
    if src.dim() == 1:
        src = src.unsqueeze(0)  # [1, S_src]
    src = src.to(device)

    B = src.size(0)

    # Step 1: Encode source (done ONCE)
    from masks import make_src_mask
    src_mask = make_src_mask(src, pad_idx).to(device)
    memory = model.encode(src, src_mask)   # [B, S_src, d_model]

    # Step 2: Initialize decoder input with SOS
    # decoder_input starts as [[SOS], [SOS], ...] for each sequence in batch
    decoder_input = torch.full((B, 1), sos_idx, dtype=torch.long, device=device) # shape: [B, 1]

    # Track which sequences have finished (produced EOS)
    finished = torch.zeros(B, dtype=torch.bool, device=device)

    # Step 3: Autoregressively generate tokens
    for _ in range(max_len):
        # Run decoder with current decoder_input
        # decoder_input shape: [B, t] where t grows each step
        logits = model.decode(decoder_input, memory, memory_mask=src_mask) # logits shape: [B, t, vocab_size]

        # Take logits for the LAST position only (we just want the next token)
        next_logits = logits[:, -1, :]   # [B, vocab_size]

        # Greedy: pick the token with the highest logit
        next_token = next_logits.argmax(dim=-1)  # [B]

        # For already-finished sequences, replace with PAD (no new meaningful tokens)
        next_token = next_token.masked_fill(finished, pad_idx)

        # Append next token to decoder input, # shape: [B, t+1]
        decoder_input = torch.cat(
            [decoder_input, next_token.unsqueeze(1)], dim=1
        )

        # Mark sequences that just produced EOS as finished
        finished = finished | (next_token == eos_idx)

        # Early stop: if all sequences are finished, we're done
        if finished.all():
            break

    # Remove the SOS token at the beginning (it was only an input cue)
    # Return shape: [B, S_out] where S_out = number of tokens generated
    return decoder_input[:, 1:]


@torch.no_grad()
def beam_search_decode(
    model: torch.nn.Module,
    src: torch.Tensor,
    beam_size: int = 4,
    max_len: int = cfg.max_seq_len,
    sos_idx: int = cfg.SOS_IDX,
    eos_idx: int = cfg.EOS_IDX,
    pad_idx: int = cfg.PAD_IDX,
    length_penalty: float = 0.6,
    device: torch.device = None
) -> List[torch.Tensor]:
    """
    Beam search decoding for a single source sequence.

    Beam search keeps the top `beam_size` most likely sequences at each step, exploring multiple hypotheses before committing to an output.

    LENGTH PENALTY (from the paper, alpha=0.6):
    ---------------------------------------
    Without length penalty, beam search tends to prefer shorter sequences
    because shorter = fewer terms in the log-prob product = less negative and close to 0.

    The paper uses:
      score = log_prob(hypothesis) / length_penalty_fn(length)
    where:
      length_penalty_fn(l) = ((5 + l) / (5 + 1))^alpha

    Higher alpha -> stronger penalty for short sequences -> longer outputs.
    alpha=0 -> no penalty (standard beam search).

    Parameters
    ----------
    model : Transformer (in eval mode)
    src : shape [S_src] or [1, S_src], Single source sequence.
    beam_size : int, Number of beams to maintain (k in top-k).
    max_len : int
    sos_idx, eos_idx, pad_idx : int
    length_penalty : float (aplha in the paper, 0.6)
    device : torch.device

    Returns
    -------
    hypotheses : List[torch.Tensor]
        List of `beam_size` generated sequences (best first, by score). Each tensor has shape [S_out].
    """
    if device is None:
        device = next(model.parameters()).device

    if src.dim() == 1:
        src = src.unsqueeze(0)   # [1, S_src]
    src = src.to(device)

    from masks import make_src_mask

    # Step 1: Encode source
    src_mask = make_src_mask(src, pad_idx).to(device)
    memory = model.encode(src, src_mask)   # [1, S_src, d_model]

    # Initialize beam
    # Each beam is: (cumulative log-prob, sequence_tensor)
    # Start: one beam with just SOS, log-prob = 0
    beams = [(0.0, torch.tensor([sos_idx], dtype=torch.long, device=device))]

    completed = []  # finished beams (produced EOS)

    # Expand each beam step by step
    for step in range(max_len):
        # Collect all candidates from expanding each current beam
        candidates = []

        for score, seq in beams:
            if seq[-1].item() == eos_idx:
                # Already finished - add to completed and don't expand
                completed.append((score, seq))
                continue

            # Run decoder with this beam's sequence # seq shape: [t] -> unsqueeze to [1, t] (batch of 1)
            decoder_input = seq.unsqueeze(0)   # [1, t]
            logits = model.decode(decoder_input, memory, memory_mask=src_mask) # logits: [1, t, vocab_size]

            # Take last position
            last_logits = logits[0, -1, :]   # [vocab_size]
            log_probs = F.log_softmax(last_logits, dim=-1)   # [vocab_size]

            # Get top-k tokens (pruning to beam_size most promising)
            top_log_probs, top_indices = log_probs.topk(beam_size)

            for log_p, idx in zip(top_log_probs, top_indices):
                new_score = score + log_p.item()
                new_seq = torch.cat([seq, idx.unsqueeze(0)])
                candidates.append((new_score, new_seq))

        if not candidates:
            break

        # Score with length penalty
        def length_penalty_fn(length):
            return ((5 + length) / 6.0) ** length_penalty

        def scored(candidate):
            s, seq = candidate
            return s / length_penalty_fn(len(seq))

        # Keep top beam_size candidates
        candidates.sort(key=scored, reverse=True)
        beams = candidates[:beam_size]

        # If all beams are finished, stop
        if all(seq[-1].item() == eos_idx for _, seq in beams):
            completed.extend(beams)
            beams = []
            break

    # Add any remaining non-finished beams to completed
    completed.extend(beams)

    # Sort by score with length penalty, best first
    completed.sort(key=lambda x: x[0] / length_penalty_fn(len(x[1])), reverse=True)

    # Return sequences without SOS token
    return [seq[1:] for _, seq in completed[:beam_size]]


def decode_tokens(token_ids: torch.Tensor, id_to_token: dict = None) -> str:
    """
    Convert a list of token IDs to a human-readable string.
    Parameters
    ----------
    token_ids : torch.Tensor or list of int
    id_to_token : dict {int: str} or None
        If None, just prints the IDs as numbers.

    Returns
    -------
    text : str
    """
    ids = token_ids.tolist() if isinstance(token_ids, torch.Tensor) else token_ids

    # Remove EOS and PAD tokens
    ids = [i for i in ids if i not in (cfg.EOS_IDX, cfg.PAD_IDX)]

    if id_to_token is not None:
        return ' '.join(id_to_token.get(i, f'<{i}>') for i in ids)
    else:
        return ' '.join(str(i) for i in ids)

Overwriting inference.py
